In [1]:
#Import#

# Core Cellpose modules
from cellpose import models, core, io, plot  # models: neural networks, core: GPU utils,
                                              # io: file I/O, plot: visualization
from cellpose import train                    # Training utilities
from cellpose import metrics                  # Evaluation metrics (AP, IoU, etc.)

# Standard libraries
import numpy as np                            # Numerical operations on arrays/masks
from pathlib import Path                      # Cross-platform file path handling
from tqdm import trange                       # Progress bars for loops (not used in this snippet)
import matplotlib.pyplot as plt               # Plotting and visualization

# Enable Cellpose logging to see progress during training/inference
# Outputs info like: "processing image 1/100", timing info, etc.
io.logger_setup()

# Verify GPU is available - critical for reasonable processing times
# Cellpose on CPU is ~10-50x slower than GPU
if core.use_gpu() == False:
    raise ImportError("No GPU access, change your runtime")
    # In Colab: Runtime -> Change runtime type -> GPU
    # Locally: Ensure CUDA and PyTorch GPU are properly installed

2025-12-19 14:32:57,392 [INFO] WRITING LOG OUTPUT TO C:\Users\v.aubry\.cellpose\run.log
2025-12-19 14:32:57,394 [INFO] 
cellpose version: 	4.0.7.dev14+gd95f58a66 
platform:       	win32 
python version: 	3.10.18 
torch version:  	2.8.0+cu128
2025-12-19 14:32:57,944 [INFO] ** TORCH CUDA version installed and working. **


In [2]:
#Data#

# Path to the microscopy dataset on network drive
# This appears to be a U2OS cell experiment with multiple stains:
# gH2AX (488nm), 53BP1 (568nm), EdU (647nm) - we're using DAPI channel for nuclear segmentation
DAPI_dir = r'Y:\Group Members\Valentin Aubry\01_Data\Final_Test'

# Validate the directory exists (network drives can disconnect)
if not Path(DAPI_dir).exists():
    raise FileNotFoundError("directory does not exist")

# Suffix pattern for segmentation mask files (used by Cellpose when saving)
pred_ext = "_seg.npy"


# Find all DAPI images in the directory
# Filter specifically for files containing "DAPI SEM.tif" in their name
# "SEM" likely refers to a specific image processing/acquisition mode
files = [f for f in Path(DAPI_dir).glob("*") if "DAPI SEM.tif" in f.name]

# Validate that we found images to process
if(len(files) == 0):
    raise FileNotFoundError("no files found, did you specify the correct folder and extension?")
else:
    print(f"{len(files)} files in DAPI folder:")

# Print all found filenames for verification
for f in files:
    print(f.name)


# Create a Path object and collect all DAPI images as a list
# This is functionally similar to the 'files' variable above but stored differently
DAPI_dir_path = Path(DAPI_dir)
DAPI_list = list(DAPI_dir_path.glob('*DAPI SEM.tif'))
# DAPI_list contains full Path objects, useful for io.imread() and saving masks

5 files in DAPI folder:
ATR3_24h--W00056--P00001--Z00000--T00000--DAPI SEM.tif
ATR3_24h--W00056--P00014--Z00000--T00000--DAPI SEM.tif
ATR3_8h--W00053--P00001--Z00000--T00000--DAPI SEM.tif
ATR4_8h--W00065--P00009--Z00000--T00000--DAPI SEM.tif
UT_8h--W00017--P00034--Z00000--T00000--DAPI SEM.tif


In [3]:
# Using Cellpose on the previously defined pictures

# Load all DAPI images into memory as numpy arrays
# io.imread() handles various formats (tif, png, etc.) and returns numpy arrays
# NOTE: Variable shadowing - 'DAPI_dir_path' in comprehension shadows the outer variable
# Consider renaming to 'img_path' for clarity:
# imgs = [io.imread(str(img_path)) for img_path in DAPI_list]
imgs = [io.imread(str(DAPI_dir_path)) for DAPI_dir_path in DAPI_list]

# Load your custom-trained DAPI segmentation model
# This model was trained earlier using the training notebook
model = models.CellposeModel(
    gpu=True,
    pretrained_model=r'C:\Users\v.aubry\Desktop\Microscopy_Data\Jupyter_Notebook\models\DAPI_model'
)


# Run segmentation on all images
masks, flows, styles = model.eval(
    imgs,
    diameter=None,      # None = let model auto-estimate cell diameter per image
                        # Set a fixed value (e.g., 30) if cells are uniform size
    channels=[0, 0],    # [cytoplasm_channel, nucleus_channel]
                        # [0,0] = grayscale (single channel, no nucleus hint)
                        # [2,1] = RGB image with red=cytoplasm, green=nucleus
                        # [1,0] = green channel only
    batch_size=8        # Images processed simultaneously on GPU
                        # Increase if GPU has memory headroom, decrease if OOM errors
)
# Returns:
# - masks: list of 2D arrays, each pixel labeled with cell ID (0 = background)
# - flows: list of flow fields (used internally by Cellpose for cell separation)
# - styles: list of style vectors (learned representations, rarely needed)


# Save segmentation results alongside original images
# For each image, creates a _seg.npy file containing masks, flows, and metadata
# e.g., "image_DAPI SEM.tif" -> "image_DAPI SEM_seg.npy"
io.masks_flows_to_seg(imgs, masks, flows, DAPI_list, channels=None)


'''
Performance benchmark:
735 Pictures: 2 hours 55 minutes processing + ~10 min saving
Approximately 14 seconds per image on average
'''

2025-12-19 14:33:59,951 [INFO] ** TORCH CUDA version installed and working. **
2025-12-19 14:33:59,953 [INFO] >>>> using GPU (CUDA)
2025-12-19 14:34:03,225 [INFO] >>>> loading model C:\Users\v.aubry\Desktop\Microscopy_Data\Jupyter_Notebook\models\DAPI_model
2025-12-19 14:34:45,481 [WARNING] channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
2025-12-19 14:35:56,655 [INFO] 100%|##########| 5/5 [01:11<00:00, 14.23s/it]


'\nPerformance benchmark:\n735 Pictures: 2 hours 55 minutes processing + ~10 min saving\nApproximately 14 seconds per image on average\n'